# Critical assessment — DSA5206 Type-2 project
Separate assessment of the Duong reproduction. No reproduction files are overwritten. All generated outputs reside under `/content/duong-critical-assessment-results/`.

**Evidence policy:** this notebook contains no precomputed experimental claims. Execute with the real historical images, metadata and expert scores; do not reconstruct predictions from figures. Code-verified discrepancies below are distinguished from dataset-dependent findings and unverified explanations.

**Colab:** clone branch `critical-assessment` (once published), open this notebook, run installation, mount Drive if necessary, configure the same data paths as the improved reproduction, then Run all. CPU works; GPU is not required. Only dependency installation and model inference require network access. See `CRITICAL_ASSESSMENT.md` for execution details.


In [ ]:
# Colab: run once. Restart the runtime if prompted, then continue below.
%pip install torch==2.5.1 torchvision==0.20.1 torchxrayvision==1.4.0 numpy==1.26.4 pandas==2.2.3 scipy==1.14.1 scikit-learn==1.5.2 scikit-image==0.24.0 matplotlib==3.9.2


In [ ]:
# Optional Drive mount:
# from google.colab import drive
# drive.mount('/content/drive')
import os, sys, json, platform, importlib.metadata
from pathlib import Path
REPO_DIR = Path(os.environ.get('DUONG_REPO_DIR','/content/nus_DSA5206_GROUPPROJECT')).resolve()
DATA_DIR = Path(os.environ.get('DUONG_DATA_DIR','/content/drive/MyDrive/duong-data')).resolve()
IMAGE_DIR = Path(os.environ.get('DUONG_IMAGE_DIR',str(DATA_DIR / 'images')))
METADATA_CSV = Path(os.environ.get('DUONG_METADATA_CSV',str(DATA_DIR / 'metadata_2020.csv')))
SCORES_CSV = Path(os.environ.get('DUONG_SCORES_CSV',str(DATA_DIR / 'covid-severity-scores.csv')))
OUTPUT_DIR = Path('/content/duong-critical-assessment-results/').resolve()
assert OUTPUT_DIR != REPO_DIR and REPO_DIR not in OUTPUT_DIR.parents
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(REPO_DIR))
os.chdir(OUTPUT_DIR)
# Read-only source of earlier critical-assessment caches. No files are written here.
CACHE_DIR = Path(os.environ.get('DUONG_CRITICAL_CACHE_DIR',str(OUTPUT_DIR)))
# Improved-reproduction legacy caches have no array integrity manifest. They are
# detected for transparency but regenerated rather than presumed to match data.
REPRO_RESULTS_DIR = Path(os.environ.get('DUONG_REPRO_RESULTS_DIR','/content/duong-improved-results'))
print('Legacy arrays present:', sorted(p.name for p in REPRO_RESULTS_DIR.glob('duong_d_*.npy')))
assert IMAGE_DIR.is_dir() and METADATA_CSV.is_file() and SCORES_CSV.is_file(), 'Supply real source data first'


## 1. Data, row alignment and provenance
Use the same PA-view loader, filename join, joint nonmissing-label mask and row order as Table 1. Group IDs retain the loader's original representation. Require 94 scored images; this count alone cannot establish historical cohort identity. Record hashes of every scored image as well as metadata, scores and checkpoint. Cached features are accepted only when the full fingerprint and payload checksum match; missing/incompatible legacy caches are regenerated from actual pixels. No labels are used to fit the frozen feature extractor.


In [ ]:
import numpy as np, pandas as pd, torch, torchvision, torchxrayvision as xrv
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from critical_methods import (sha256_file, fingerprint, evaluate_features, exact_splits,
                             stability_summary, tolerance_scores, SPLIT_CONFIG)
from duong_methods import saliency_pair
from IPython.display import display
transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),xrv.datasets.XRayResizer(224)])
dmerge = xrv.datasets.COVID19_Dataset(imgpath=str(IMAGE_DIR),csvpath=str(METADATA_CSV),transform=transform,views=['PA'])
scores = pd.read_csv(SCORES_CSV,skiprows=5).set_index('filename')
assert scores.index.is_unique and dmerge.csv.filename.is_unique
assert dmerge.csv.patientid.notna().all()
dmerge.csv = dmerge.csv.set_index('filename').join(scores,on='filename').reset_index()
scored_indices = np.flatnonzero((dmerge.csv.geographic_mean.notna() & dmerge.csv.opacity_mean.notna()).to_numpy())
scored_metadata = dmerge.csv.iloc[scored_indices].copy().reset_index(drop=True)
assert len(scored_metadata)==94
assert scored_metadata.filename.is_unique
scored_dataset = torch.utils.data.Subset(dmerge,scored_indices.tolist())
# Reuse already-downloaded reproduction weights where available, without modifying them.
weight_dir = OUTPUT_DIR / 'weights'
weight_dir.mkdir(parents=True,exist_ok=True)
# Read source cache only; copy the exact expected checkpoint to assessment output.
import shutil
from urllib.parse import urlparse
weight_name = Path(urlparse(xrv.models.model_urls['all']['weights_url']).path).name
source_weight = REPRO_RESULTS_DIR / 'weights' / weight_name
if source_weight.is_file() and not (weight_dir / weight_name).exists():
    shutil.copyfile(source_weight,weight_dir / weight_name)
nn_model = xrv.models.DenseNet(weights='all',cache_dir=str(weight_dir)).eval()
nn_model.op_threshs = None
checkpoint = Path(nn_model.weights_filename_local)
image_hashes = {f:sha256_file(IMAGE_DIR / f) for f in scored_metadata.filename}
provenance = {
 'schema':1, 'python':platform.python_version(),
 'packages':{p:importlib.metadata.version(p) for p in ['torch','torchvision','torchxrayvision','numpy','pandas','scipy','scikit-learn','scikit-image','matplotlib']},
 'metadata_sha256':sha256_file(METADATA_CSV), 'scores_sha256':sha256_file(SCORES_CSV),
 'checkpoint_sha256':sha256_file(checkpoint), 'weights_alias':'all',
 'pathologies':list(nn_model.pathologies), 'ordered_filenames':scored_metadata.filename.tolist(),
 'scored_image_sha256':image_hashes, 'split_config':SPLIT_CONFIG,
 'feature_pipeline':'PA; normalize maxval255; center crop; resize224; eval; post-ReLU pooled1024; pre-sigmoid18',
 'assessment_code_sha256':sha256_file(REPO_DIR / 'critical_methods.py'),
 'saliency_code_sha256':sha256_file(REPO_DIR / 'duong_methods.py')}
run_fingerprint = fingerprint(provenance)
(OUTPUT_DIR / 'provenance.json').write_text(json.dumps(provenance,indent=2))
scored_metadata.to_csv(OUTPUT_DIR / 'ordered_scored_manifest.csv',index=False)
print('Real scored images:',len(scored_metadata),'unique patients:',scored_metadata.patientid.nunique())
print('Fingerprint:',run_fingerprint,'checkpoint:',str(checkpoint))


In [ ]:
cache_npz = CACHE_DIR / 'assessment_features.npz'
cache_manifest = CACHE_DIR / 'assessment_features_manifest.json'
reused = False
if cache_npz.is_file() and cache_manifest.is_file():
    saved = json.loads(cache_manifest.read_text())
    if saved.get('fingerprint') == run_fingerprint and saved.get('sha256') == sha256_file(cache_npz):
        with np.load(cache_npz,allow_pickle=False) as arr:
            d_feats, d_preds = arr['features'].copy(),arr['logits'].copy()
        reused = True
if not reused:
    all_features,all_logits = [],[]
    torch.set_num_threads(2)
    with torch.no_grad():
        for batch in torch.utils.data.DataLoader(scored_dataset,batch_size=8,shuffle=False,num_workers=0):
            x=batch['img']
            feats=F.adaptive_avg_pool2d(F.relu(nn_model.features(x),inplace=False),(1,1)).flatten(1)
            all_features.append(feats.cpu().numpy())
            all_logits.append(nn_model(x).cpu().numpy())
    d_feats,d_preds = np.concatenate(all_features),np.concatenate(all_logits)
assert d_feats.shape==(94,1024) and d_preds.shape==(94,18)
assert np.isfinite(d_feats).all() and np.isfinite(d_preds).all()
# Saving is confined to new assessment outputs, even when reading another cache.
np.savez(OUTPUT_DIR / 'assessment_features.npz',features=d_feats,logits=d_preds)
(OUTPUT_DIR / 'assessment_features_manifest.json').write_text(json.dumps({
 'fingerprint':run_fingerprint,'sha256':sha256_file(OUTPUT_DIR / 'assessment_features.npz')},indent=2))
print('Feature source:', 'verified numerical cache' if reused else 'real image inference')
pathologies=list(xrv.datasets.default_pathologies)
assert list(nn_model.pathologies)==pathologies
opacity_idx=pathologies.index('Lung Opacity')
feature_sets={
 'Intermediate network features':d_feats, 'No data':np.zeros((94,1)),
 '18 outputs':d_preds,
 '4 outputs':d_preds[:,[pathologies.index(p) for p in ['Lung Opacity','Pneumonia','Infiltration','Consolidation']]],
 'lung opacity output':d_preds[:,[opacity_idx]]}


## 2. Exact Table 1 splits and actual out-of-sample predictions
For each seed 0…49: `GroupShuffleSplit(train_size=0.5, test_size=0.5, random_state=seed)` on the unchanged ordered rows and patient IDs. This is a split of **groups**, so image counts need not be exactly 47/47. Every feature/target combination uses the same indices. OLS is fitted solely on training rows; thresholds, features and seeds are fixed before evaluation. Frozen checkpoint pretraining-overlap provenance cannot be certified from this repository.

Export all 50 split memberships and assert zero overlapping patient IDs before fitting. Cached summary tables alone cannot recover individual residuals; regenerate OLS predictions from validated real features instead. Regression is inexpensive relative to image inference.


In [ ]:
metrics, predictions, split_audits = evaluate_features(feature_sets,scored_metadata)
assert len(metrics)==500 and len(split_audits)==50
assert all(a['overlap_count']==0 for a in split_audits)
(OUTPUT_DIR / 'split_membership.json').write_text(json.dumps(split_audits,indent=2,default=lambda v:v.item()))
metrics.to_csv(OUTPUT_DIR / 'split_metrics.csv',index=False)
predictions.to_csv(OUTPUT_DIR / 'heldout_predictions.csv',index=False)
audit_table=pd.DataFrame([{'seed':a['seed'],'train_images':len(a['train_indices']),
 'test_images':len(a['test_indices']),'train_patients':len(set(a['train_patient_ids'])),
 'test_patients':len(set(a['test_patient_ids'])),'overlap_count':a['overlap_count']} for a in split_audits])
audit_table.to_csv(OUTPUT_DIR / 'patient_overlap_audit.csv',index=False)
display(audit_table)
print('CONFIRMED IN THIS RUN: all 50 splits are patient-disjoint.')


## 3. Original versus corrected tolerance metric
The released code counts `(prediction − truth) < 0.7`, including arbitrarily severe underpredictions. The corrected definition counts `abs(prediction − truth) < 0.7`. Their exact difference is the fraction of held-out predictions with error **≤ −0.7** (the negative exact-threshold case is included). This auxiliary metric is not a reported Table 1 paper metric.

Compare both severity targets and all five feature sets on the same predictions and splits. Report the mean of 50 split accuracies; do not pool repeated appearances of the same image as independent observations. These descriptive differences are not confidence intervals or clinical accuracy estimates.


In [ ]:
accuracy_summary=metrics.groupby(['target','features'])[['one_sided_accuracy','absolute_accuracy','excess_accuracy']].agg(['mean','std','min','max'])
accuracy_summary.to_csv(OUTPUT_DIR / 'tolerance_comparison_summary.csv')
display(accuracy_summary)
np.testing.assert_allclose(metrics.one_sided_accuracy-metrics.absolute_accuracy,metrics.excess_accuracy,atol=1e-15)
fig,axes=plt.subplots(1,2,figsize=(15,5))
for ax,target in zip(axes,['Geographic Extent','Opacity']):
    avg=metrics[metrics.target==target].groupby('features')[['one_sided_accuracy','absolute_accuracy']].mean()
    avg.plot.bar(ax=ax,ylim=(0,1),rot=30)
    ax.set_title(target+' — same 50 held-out splits');ax.set_ylabel('Mean split accuracy')
fig.tight_layout();fig.savefig(OUTPUT_DIR / 'tolerance_accuracy_comparison.png',dpi=180);plt.show()


## 4. Stability across repeated patient-grouped splits
Display mean, sample standard deviation, median, minimum, maximum, 5th/95th empirical percentiles and number of defined values. Boxplots and per-seed traces describe sensitivity to train/test composition. The 50 splits overlap in patients: they are dependent resamples, **not 50 independent replications**. No naive t-test, standard-error confidence interval or claim of external generalization is made. Baseline Pearson correlation stays N/A because predictions are constant.


In [ ]:
summary=stability_summary(metrics)
summary.to_csv(OUTPUT_DIR / 'metric_stability_summary.csv');display(summary)
metric_names=['MAE','MSE','R2','Pearson']
for target in ['Geographic Extent','Opacity']:
    subset=metrics[metrics.target==target]
    fig,axes=plt.subplots(2,4,figsize=(22,9))
    for j,metric in enumerate(metric_names):
        names=[];values=[]
        for name,g in subset.groupby('features',sort=False):
            series=g.sort_values('seed')[metric]
            axes[1,j].plot(g.sort_values('seed').seed,series,label=name,alpha=.75)
            if series.notna().any(): names.append(name);values.append(series.dropna().to_numpy())
        axes[0,j].boxplot(values,labels=names,showfliers=True)
        axes[0,j].tick_params(axis='x',rotation=65)
        axes[0,j].set_title(metric+' distribution')
        axes[1,j].set_title(metric+' by fixed split seed');axes[1,j].set_xlabel('Seed')
    axes[1,0].legend(fontsize=8)
    fig.suptitle(target+' — dependent split variability; baseline Pearson undefined')
    fig.tight_layout();fig.savefig(OUTPUT_DIR / (target.replace(' ','_')+'_stability.png'),dpi=180);plt.show()


## 5. Documented discrepancies (source inspection, independent of this run)

| Topic | Paper / supplied reference | Released implementation and reproduction | Confirmed conclusion | Unverified explanation / limit |
|---|---|---|---|---|
| Tolerance metric | Table 1 reports Pearson, R², MAE, MSE; does not define the auxiliary `<0.7 ACC` | Author cell 34 and original reproduction cell 16 use signed error `<0.7` | One-sided metric incorrectly rewards large underpredictions; comparison above quantifies the effect using actual predictions | This does not establish an error in the paper's reported Table 1 metrics |
| Figure 3 sample count | Paper Figure 3 says 208; author notebook cell 23 saved output is `(199,2)` | Improved reproduction retains 208; four pre-sigmoid outputs: Lung Opacity, Pneumonia, Infiltration, Consolidation | Saved reference count and published description disagree; no nine-image removal is justified | Dataset revisions, availability or loader filtering are plausible explanations, not established causes; no 199-image manifest is supplied |
| Figure 4 saliency | §2.6/Figure 4 describe gradient with 5×5 Gaussian blur | Author cell 82 and original reproduction cell 25 square gradients then sigma 5, truncate 3.5 (37×37), nearest boundary | Released code differs from the stated kernel and gradient transformation | Paper omits sigma, gradient sign treatment and boundary handling; B below is one explicit interpretation |

Sources: repository paper DOI 10.7759/cureus.9448, `xray_covid_eval.ipynb`, `Duong_Reproduction_Code.ipynb`, `Duong_Reproduction_Improved.ipynb`. The improved code corrects the metric while originals remain intact. Changes in regression seed (author qualitative seed 49 vs reproduction seed 0) also prevent claiming identical qualitative results; this assessment preserves seed 0 for the selected saliency cases.


## 6. Paired saliency analysis on identical held-out images
Use the reproduction's fixed seed-0 geographic-extent regression and deliberate case-selection rules. Cases are selected after evaluation to illustrate successes and failures; they are **not representative**, and they do not influence fitted models, hyperparameters or reported metrics. A threshold category can be empty on a different dataset; do not replace it silently or tune the seed.

- **A, released code:** squared input gradient; sigma=(5,5), truncate=3.5, nearest-edge boundary; effective 37×37 support.
- **B, paper-description interpretation:** signed input gradient; normalized discrete 5×5 Gaussian (sigma=1 pixel), reflect half-sample-symmetric boundary. These choices are not fully specified in the paper.

Calculate one gradient per image from the same checkpoint, coefficient and intercept, then transform it using A and B. All originals use [-1024,1024] grayscale limits and all overlays alpha 0.5. A has one global nonnegative scale across the four cases; B has one global symmetric scale across them. Separate colorbars are necessary because their units differ. Comparing color intensity across A/B is not a quantitative localization test. No radiologist segmentation ground truth is available to score localization.


In [ ]:
SALIENCY_SEED = 0
#Figure 4 的代码
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from skimage.filters import gaussian
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit

# ---------------------------
# 1. 重新准备测试集预测结果
# ---------------------------
preds_94 = d_preds
meta_94 = scored_metadata

opacity_idx = xrv.datasets.default_pathologies.index("Lung Opacity")

X = preds_94[:, [opacity_idx]]
y = meta_94["geographic_mean"].astype(float).to_numpy()
groups = meta_94["patientid"]

splitter = GroupShuffleSplit(
    train_size=0.5,
    test_size=0.5,
    random_state=SALIENCY_SEED
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

geo_model = LinearRegression()
geo_model.fit(X[train_idx], y[train_idx])

y_true = y[test_idx]
y_pred = geo_model.predict(X[test_idx])
errors = y_pred - y_true
abs_errors = np.abs(errors)

test_df = pd.DataFrame({
    "global_idx": test_idx,
    "filename": meta_94.iloc[test_idx]["filename"].values,
    "ground_truth": y_true,
    "prediction": y_pred,
    "error": errors,
    "abs_error": abs_errors
}).reset_index(drop=True)

assert set(meta_94.iloc[train_idx].patientid).isdisjoint(set(meta_94.iloc[test_idx].patientid))
print("Test set size:", len(test_df))

# ---------------------------
# 2. 自动选4个代表样本
# ---------------------------

# (a) 严重且预测较好：GT高，误差小
cand_a = test_df[test_df["ground_truth"] >= 4].sort_values(
    ["abs_error", "ground_truth"], ascending=[True, False]
).iloc[0]

# (b) 轻症且预测较好：GT低，误差小
cand_b = test_df[test_df["ground_truth"] <= 1].sort_values(
    ["abs_error", "ground_truth"], ascending=[True, True]
).iloc[0]

# (c) 明显高估：GT低，但预测偏高最多
cand_c = test_df[test_df["ground_truth"] <= 2].sort_values(
    "error", ascending=False
).iloc[0]

# (d) 明显低估：GT高，但预测偏低最多
cand_d = test_df[test_df["ground_truth"] >= 4].sort_values(
    "error", ascending=True
).iloc[0]

selected_cases = [
    ("(a) relatively correct / severe", cand_a),
    ("(b) relatively correct / mild", cand_b),
    ("(c) overestimation", cand_c),
    ("(d) underestimation", cand_d),
]

selected_df = pd.DataFrame([
    {
        "Case": name,
        "filename": row["filename"],
        "ground_truth": row["ground_truth"],
        "prediction": row["prediction"],
        "error": row["error"]
    }
    for name, row in selected_cases
])

display(selected_df)

# ---------------------------
# 3. 定义 saliency 计算函数
# ---------------------------
def make_saliency_for_idx(global_idx):
    sample = scored_dataset[int(global_idx)]
    img = sample["img"]

    x = torch.as_tensor(img, dtype=torch.float32).unsqueeze(0).clone().detach()
    x.requires_grad_(True)

    nn_model.eval()
    nn_model.zero_grad()

    outputs = nn_model(x)
    # Cache/checkpoint/input consistency: this prediction must match extraction.
    np.testing.assert_allclose(outputs.detach().numpy()[0],d_preds[int(global_idx)],rtol=1e-4,atol=1e-4)
    lung_opacity = outputs[:, opacity_idx]

    theta = float(geo_model.coef_[0])
    bias = float(geo_model.intercept_)

    predicted_extent = theta * lung_opacity + bias

    grads = torch.autograd.grad(predicted_extent.sum(), x)[0][0, 0].detach().cpu().numpy()

    a, b = saliency_pair(grads)
    return img[0], a, b, predicted_extent.item()



In [ ]:
panels = [make_saliency_for_idx(int(row['global_idx'])) for _, row in selected_cases]
a_max = max(float(a.max()) for _, a, b, pred in panels)
b_max = max(float(np.abs(b).max()) for _, a, b, pred in panels)
fig, axes = plt.subplots(4, 3, figsize=(15, 16))
for j, ((case, row), (img, a, b, pred)) in enumerate(zip(selected_cases, panels)):
    for ax in axes[j]:
        ax.imshow(img, cmap='gray', vmin=-1024, vmax=1024)
        ax.axis('off')
    axes[j,0].set_title(f"Deliberately selected: {case}\n{row['filename']}\nGT={row['ground_truth']:.1f}, Pred={pred:.2f}")
    im_a=axes[j,1].imshow(a, cmap='hot', alpha=.5, vmin=0, vmax=a_max)
    im_b=axes[j,2].imshow(b, cmap='coolwarm', alpha=.5, vmin=-b_max, vmax=b_max)
    axes[j,1].set_title('A: released code / squared gradient')
    axes[j,2].set_title('B: paper-description interpretation / signed gradient')
    fig.colorbar(im_a, ax=axes[j,1], fraction=.046)
    fig.colorbar(im_b, ax=axes[j,2], fraction=.046)
plt.tight_layout()
plt.savefig('Critical_Saliency_Methods_AB.png', dpi=300, bbox_inches='tight')
selected_df.to_csv('critical_selected_cases.csv', index=False)
plt.show()

# Store coefficients and raw arrays so visual normalization can be audited.
(OUTPUT_DIR / 'saliency_regression.json').write_text(json.dumps({
 'seed':SALIENCY_SEED,'coefficient':float(geo_model.coef_[0]),'intercept':float(geo_model.intercept_),
 'checkpoint_sha256':provenance['checkpoint_sha256'],
 'selected_filenames':selected_df.filename.tolist(),'A_vmin':0,'A_vmax':a_max,
 'B_vmin':-b_max,'B_vmax':b_max,'overlay_alpha':.5},indent=2))
for i,(img,a,b,pred) in enumerate(panels):
    np.savez(OUTPUT_DIR / f'saliency_case_{i}.npz',image=img,method_A=a,method_B=b,prediction=pred)


## 7. Critical discussion and evidence boundaries

**Sample size and stability.** The effective unit of generalization is a patient, not an image: repeated time points are correlated. Ninety-four labelled images constrain how much a 1024-feature OLS model can learn; many train splits have fewer images than features, making fits underdetermined. Correlated resampling and selection of favourable illustrations must not be mistaken for robust prospective evidence. Use the generated descriptive statistics to discuss observed variability, without treating their percentiles as population confidence intervals.

**External validation.** These splits test internal discrimination within a small retrospective convenience cohort. They cannot establish transportability across hospitals, devices, acquisition views or pandemic periods. Dataset/site confounding and undocumented pretraining overlap are hypotheses requiring provenance and external cohorts, not demonstrated explanations for observed performance. No survival-prediction validation is performed here.

**Label uncertainty.** Expert-averaged scores are ordinal-derived annotations, not error-free physiological measurements. The paper reports inter-rater agreement (opacity 0.45, extent 0.71). Rounding, disagreement and heterogeneous disease time points limit interpretation of small residual differences. Rater-level uncertainty cannot be estimated from mean-only CSV columns. The auxiliary 0.7 threshold is fixed for comparison, not established as a clinically validated decision threshold.

**Clinical interpretability.** Gradient maps measure local model sensitivity, not causal evidence or validated lesion localization. Squaring, smoothing, boundary handling and color scales change the explanation. A/B visual differences cannot identify the correct method or prove clinical benefit. A severity regressor, even with strong internal correlations, does not justify care escalation without prospective external evaluation, calibration, subgroup analysis and assessment of harms.

**Reporting:** the discrepancy table is confirmed by source inspection; numerical comparisons and zero-overlap claims are confirmed only after successful real-data execution. The missing-nine-image explanation and historical checkpoint equivalence remain unverified. No synthetic values are substituted into the report. Keep generated outputs alongside provenance when presenting the assessment.
